# Lecture 2: The Ingestion Bottleneck and Streaming

**Goal:** Build a high-throughput, infinitely streaming DataLoader that feeds a neural network from remote object storage, and understand the mathematical foundations of shuffle buffers, residual networks, and cross-entropy loss.

---

## 1. The Shuffle Buffer Mechanism

### Why Can't We Just Shuffle the Whole Dataset?
In Spark, a global `orderBy(rand())` or `reduceByKey` shuffles data by materializing the entire dataset in memory across the cluster. For a 10TB ImageNet stored on S3, this would require:
- 10TB of RAM just to hold the data
- Additional memory for the shuffle operation itself
- Network transfer of the entire dataset before training begins

This is $\mathcal{O}(D)$ memory complexity, where $D$ is dataset size — **computationally unfeasible**.

### The Sliding-Window Buffer Solution
Instead of global shuffling, we use a **reservoir-style sliding window**:
1. Fill a buffer of size $K$ with the first $K$ items from the stream
2. Randomly select one item from the buffer to yield as output
3. Fill the vacated slot with the next item from the stream
4. Repeat

This requires only $\mathcal{O}(K)$ memory regardless of dataset size $D$!

**Key insight for SGD:** Stochastic Gradient Descent doesn't require perfect uniform randomness — it only requires that consecutive minibatches are sufficiently decorrelated. A buffer of $K \geq 1000$ typically provides enough pseudo-randomness.

### Interactive Demonstration
The simulation below creates a worst-case scenario: a stream where all class-0 items arrive first, then all class-1 items, etc. Observe how increasing $K$ transforms a highly correlated stream into a approximately uniform distribution.

In [ ]:
# ============================================================================
# INTERACTIVE SHUFFLE BUFFER SIMULATOR
# ============================================================================
# This cell simulates the sliding-window shuffle buffer algorithm and
# visualizes how buffer size K affects the class distribution of yielded items.

import numpy as np                       # Array operations and random number generation
import matplotlib.pyplot as plt          # Plotting histograms
import ipywidgets as widgets             # Interactive sliders
from IPython.display import display      # Widget rendering

# Create a worst-case ordered stream: 1000 items of class 0, then 1000 of class 1, etc.
# np.repeat([0,1,...,9], 1000) produces [0,0,...,0, 1,1,...,1, ..., 9,9,...,9]
# Total: 10,000 items, perfectly sorted by class (maximum correlation)
stream_labels = np.repeat(np.arange(10), 1000)

def simulate_shuffle(buffer_size=10):
    """
    Simulates the sliding-window shuffle buffer algorithm.
    
    The algorithm works exactly like WebDataset's wds.shuffle(K):
    1. Read K items into a buffer from the sequential stream
    2. Pop a random item from the buffer (this is the "yield")
    3. Push the next stream item into the buffer to fill the gap
    4. Repeat until we've yielded enough items
    
    Args:
        buffer_size (int): The buffer capacity K. Controls the
                           randomness vs. memory tradeoff.
                           K=1 → no shuffling (sequential order)
                           K=D → perfect global shuffle (but O(D) memory)
    """
    np.random.seed(42)  # Fixed seed for reproducible demonstrations
    
    # Initialize the buffer with the first K items from the stream
    buffer = list(stream_labels[:buffer_size])
    stream_idx = buffer_size  # Pointer to the next unread item in the stream
    
    yielded = []  # Collect the items we output
    
    # Simulate yielding 1000 items from the shuffled stream
    for _ in range(1000):
        if not buffer:
            break
        # Randomly select an index in the buffer (uniform random)
        pop_idx = np.random.randint(0, len(buffer))
        # Remove and yield that item
        yielded.append(buffer.pop(pop_idx))
        # Refill the buffer with the next item from the stream
        if stream_idx < len(stream_labels):
            buffer.append(stream_labels[stream_idx])
            stream_idx += 1
            
    # --- Visualization ---
    fig, ax = plt.subplots(figsize=(8, 4))
    
    # np.bincount counts occurrences of each integer value
    # minlength=10 ensures we always have bins 0-9 even if some classes are missing
    counts = np.bincount(yielded, minlength=10)
    
    ax.bar(np.arange(10), counts, color='#2ecc71', edgecolor='black')
    
    # Red dashed line at 100 = perfect uniform distribution (1000 items / 10 classes)
    ax.axhline(100, color='r', linestyle='--', label='Perfect Uniform Target (100)')
    
    ax.set_title(f'Class Distribution of 1000 Yielded Items (Buffer K={buffer_size})')
    ax.set_ylim(0, 1000)
    ax.set_xlabel('Class Label (0-9)')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

# Build the interactive widget
# Try K=1 (no shuffle), K=100 (slight mixing), K=5000 (nearly uniform)
interact_ui = widgets.interactive(
    simulate_shuffle,
    buffer_size=widgets.IntSlider(min=1, max=5000, step=100, value=10,
                                  description='Buffer (K):')
)
display(interact_ui)

---

## 2. Optimizing the PyTorch DataLoader

### The Producer-Consumer Model
In deep learning, the **GPU** is the consumer and the **DataLoader** is the producer. If the producer can't supply data faster than the GPU processes it, the GPU idles — wasting expensive compute resources.

### Key DataLoader Parameters:
| Parameter | Purpose | Typical Value |
|-----------|---------|---------------|
| `batch_size` | Number of samples per gradient update | 64–1024 |
| `num_workers` | Background processes fetching data in parallel | 2–8 |
| `pin_memory` | Pre-allocates page-locked (pinned) CPU memory for faster GPU transfer | True (CUDA only) |
| `persistent_workers` | Keeps worker processes alive between epochs (avoids respawn overhead) | True |

### Why `num_workers > 0`?
Python's Global Interpreter Lock (GIL) prevents true multi-threading. `num_workers` spawns separate **processes** (not threads) that each independently:
1. Open HTTP connections to MinIO
2. Stream and decompress .tar bytes
3. Decode images and apply transforms
4. Place results in a shared memory queue

The main process simply dequeues batches, achieving true parallelism.

In [ ]:
# ============================================================================
# STREAMING DATALOADER CONFIGURATION
# ============================================================================
# This cell constructs a WebDataset pipeline wrapped in a PyTorch DataLoader,
# configured for maximum throughput from our MinIO object store.

import webdataset as wds                 # Streaming .tar dataset library
import torch                             # PyTorch tensor library
import time                              # Benchmarking
from torch.utils.data import DataLoader  # PyTorch's parallel data loading utility

# Brace expansion URL: {000000..000049} generates 50 shard URLs
# WebDataset will cycle through these shards, streaming each over HTTP
url = "http://127.0.0.1:9000/cifar-streaming/cifar-train-{000000..000049}.tar"

# Build the WebDataset pipeline with shuffling enabled for training
dataset = (
    wds.WebDataset(url, shardshuffle=True)  # Randomly permute shard order each epoch
                                             # This provides inter-shard randomness
    .shuffle(1000)          # Intra-shard shuffle buffer (K=1000)
                            # Provides local randomness within each shard
    .decode("rgb")          # PNG bytes → numpy float32 array (H, W, 3), range [0, 1]
    .to_tuple("png", "cls") # Package as (image_array, class_label) tuples
)

# Wrap in PyTorch DataLoader for batching and parallel loading
loader = DataLoader(
    dataset,
    batch_size=256,       # 256 samples per minibatch
                          # GPU memory constrains this: larger = faster but more VRAM
    num_workers=4,        # 4 background processes fetching data in parallel
                          # Each worker opens its own HTTP connection to MinIO
    pin_memory=True       # Pre-allocate page-locked CPU memory for async GPU transfer
                          # Only effective with CUDA GPUs (ignored on MPS/CPU)
)

print("DataLoader bound to stream successfully.")
print(f"  Batch size: 256 | Workers: 4 | Pin memory: True")
print(f"  Expected batches per epoch: ~{50000 // 256}")

---

## 3. Model Architecture: ResNet9

### The Vanishing Gradient Problem
As neural networks grow deeper, gradients computed during backpropagation get multiplied through many layers. If each layer's Jacobian has spectral norm < 1, gradients shrink exponentially:
$$\frac{\partial \mathcal{L}}{\partial x_0} = \prod_{i=1}^{L} \frac{\partial x_i}{\partial x_{i-1}} \cdot \frac{\partial \mathcal{L}}{\partial x_L} \approx 0 \quad \text{(vanishes)}$$

### The Residual Solution (He et al., 2015)
Instead of learning a direct mapping $H(x)$, residual blocks learn a **perturbation** $\mathcal{F}(x)$ added to the identity:
$$y = \mathcal{F}(x, \{W_i\}) + x$$

The key mathematical insight is in the derivative. Applying the chain rule:
$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial y} \left( \frac{\partial \mathcal{F}}{\partial x} + \mathbf{1} \right)$$

The **+1 term** (from the identity shortcut) guarantees that gradients always have a direct path back through the network, regardless of what $\mathcal{F}$ learns. This is why ResNets can scale to 1000+ layers.

### Our ResNet9 Architecture
A minimal but complete residual network for CIFAR-10 (32×32 RGB images, 10 classes):

```
Input: (N, 3, 32, 32)
  ↓ Conv2d(3→64, 3×3, pad=1) + BatchNorm + ReLU
  ↓ → (N, 64, 32, 32)
  ↓
  ├──→ ResConv1(64→64, 3×3) + ReLU + ResConv2(64→64, 3×3)  ←── F(x)
  │                                                             │
  └────────────────── identity shortcut ──────────────────────→ (+)
                                                                 ↓ ReLU
  ↓ MaxPool2d(2) → (N, 64, 16, 16)
  ↓ Reshape → (N, 64×16×16) = (N, 16384)
  ↓ Linear(16384 → 10) → (N, 10) class logits
```

In [ ]:
# ============================================================================
# RESNET9 MODEL DEFINITION
# ============================================================================
# A minimal residual network for CIFAR-10 classification.

import torch.nn as nn            # Neural network layer definitions
import torch.nn.functional as F  # Functional API (activation functions, loss functions)

class ResNet9(nn.Module):
    """
    A 9-layer residual network for 32x32 image classification.
    
    Architecture choices explained:
    - kernel_size=3, padding=1: Preserves spatial dimensions (32x32 → 32x32)
      This is the standard "same convolution" used in virtually all modern CNNs.
    - BatchNorm2d: Normalizes activations to zero mean, unit variance per channel.
      This stabilizes training and allows higher learning rates.
    - MaxPool2d(2): Halves spatial dimensions (32→16). Reduces computation and
      introduces translation invariance.
    - The residual connection (out + res) is the mathematical identity shortcut
      that enables deep gradient flow.
    """
    def __init__(self):
        super().__init__()
        
        # === Initial feature extraction ===
        # Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1)
        # Input: (N, 3, 32, 32) RGB images
        # Output: (N, 64, 32, 32) — 64 learned feature maps
        # Parameters: 3 × 64 × 3 × 3 + 64 bias = 1,792
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        
        # BatchNorm2d normalizes each of the 64 channels independently:
        #   x_normalized = (x - mean) / sqrt(variance + eps)
        # Then applies learnable affine: gamma * x_normalized + beta
        # Parameters: 64 (gamma) + 64 (beta) = 128
        self.bn1 = nn.BatchNorm2d(64)
        
        # === Residual Block (the core innovation) ===
        # Two consecutive 3×3 convolutions that learn F(x)
        # The output will be F(x) + x (residual connection)
        self.res_conv1 = nn.Conv2d(64, 64, kernel_size=3, padding=1)  # 64×64×3×3 = 36,928
        self.res_conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)  # 64×64×3×3 = 36,928
        
        # MaxPool2d(2) reduces spatial dimensions by 2x in each direction
        # (N, 64, 32, 32) → (N, 64, 16, 16)
        self.pool = nn.MaxPool2d(2)
        
        # Fully connected classification head
        # Input: 64 channels × 16 height × 16 width = 16,384 features
        # Output: 10 class logits (one per CIFAR-10 class)
        self.fc = nn.Linear(64 * 16 * 16, 10)

    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x (Tensor): Input images, shape (N, 3, 32, 32)
            
        Returns:
            Tensor: Class logits, shape (N, 10)
        """
        # === Stage 1: Initial convolution + normalization + activation ===
        # Conv2d extracts 64 feature maps from the 3 RGB channels
        # BatchNorm stabilizes the distribution of activations
        # ReLU(x) = max(0, x) introduces non-linearity
        out = F.relu(self.bn1(self.conv1(x)))  # (N, 3, 32, 32) → (N, 64, 32, 32)
        
        # === Stage 2: Residual block ===
        # This implements y = F(x, {W_i}) + x
        # where F is two convolutions with a ReLU between them
        res = self.res_conv1(out)    # First conv of F(x)
        res = F.relu(res)            # Non-linearity within F(x)
        res = self.res_conv2(res)    # Second conv of F(x)
        out = F.relu(out + res)      # THE KEY: identity shortcut + F(x), then ReLU
        
        # === Stage 3: Spatial downsampling ===
        out = self.pool(out)  # (N, 64, 32, 32) → (N, 64, 16, 16)
        
        # === Stage 4: Classification head ===
        # Flatten spatial dimensions for the linear layer.
        # .reshape() handles non-contiguous memory (from permute operations)
        # .view() would crash on non-contiguous tensors
        out = out.reshape(out.size(0), -1)  # (N, 64, 16, 16) → (N, 16384)
        
        return self.fc(out)  # (N, 16384) → (N, 10) raw logits

# Instantiate the model and print its structure
model = ResNet9()
total_params = sum(p.numel() for p in model.parameters())
print(f"ResNet9 constructed: {total_params:,} trainable parameters")

---

## 4. Active Training with PyTorch Lightning

### What is PyTorch Lightning?
PyTorch Lightning is an abstraction layer that separates **engineering code** (GPU management, distributed training, logging, checkpointing) from **research code** (model architecture, loss function, optimizer). You define:
- `training_step()`: What happens for one minibatch
- `configure_optimizers()`: Which optimizer to use
Lightning handles everything else: device placement, gradient accumulation, mixed precision, multi-GPU, etc.

### Categorical Cross-Entropy Loss
For a classification problem with $C$ classes, the model outputs a vector of **logits** $z \in \mathbb{R}^C$. The softmax function converts logits to probabilities:
$$\hat{y}_c = \frac{e^{z_c}}{\sum_{j=1}^{C} e^{z_j}}$$

The cross-entropy loss measures the "surprise" of the predicted distribution relative to the true label:
$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c) = -\log(\hat{y}_{true})$$

where $y_c$ is a one-hot vector (1 for the true class, 0 elsewhere). PyTorch's `F.cross_entropy` implements the numerically stable version using the log-sum-exp trick:
$$\mathcal{L} = -z_{true} + \log\left(\sum_{j=1}^{C} e^{z_j}\right)$$

### AdamW Optimizer
We use AdamW (Adam with decoupled Weight Decay), which maintains per-parameter adaptive learning rates using first and second moment estimates:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(momentum)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(RMSprop-style variance)}$$
$$\theta_t = \theta_{t-1} - \eta \left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_{t-1}\right)$$

where $\lambda$ is the weight decay coefficient that regularizes the model.

In [ ]:
# ============================================================================
# PYTORCH LIGHTNING TRAINING LOOP
# ============================================================================
# This cell defines the LightningModule wrapper and executes real training
# and evaluation over our streamed WebDataset pipeline.

import pytorch_lightning as pl   # High-level training framework
import torch.optim as optim      # Optimizer implementations (SGD, Adam, AdamW, etc.)

class TrafficLight(pl.LightningModule):
    """
    A PyTorch Lightning wrapper for our ResNet9 model.
    
    LightningModule replaces the manual training loop:
        for epoch in range(epochs):
            for batch in loader:
                optimizer.zero_grad()
                loss = model(batch)
                loss.backward()
                optimizer.step()
    
    with a declarative interface where you only define WHAT to compute,
    and Lightning handles HOW to execute it (device placement, logging, etc.)
    """
    def __init__(self, model):
        super().__init__()
        self.model = model  # Our ResNet9 instance

    def forward(self, x):
        """Standard forward pass — delegates to the underlying model."""
        return self.model(x)

    def training_step(self, batch, batch_idx):
        """
        Defines the computation for a single training minibatch.
        
        PyTorch Lightning automatically handles:
        - Moving data to the correct device (CPU/GPU/TPU)
        - Calling loss.backward()
        - Calling optimizer.step()
        - Calling optimizer.zero_grad()
        - Gradient clipping (if configured)
        
        Args:
            batch: Tuple of (images, labels) from the DataLoader
            batch_idx: Index of this batch within the epoch
            
        Returns:
            loss: Scalar tensor — Lightning will backpropagate this automatically
        """
        images, labels = batch
        
        # WebDataset may return uint8 [0, 255] or float32 [0, 1] depending
        # on the decode mode. We normalize to float32 [0, 1] for the network.
        if images.dtype == torch.uint8:
            images = images.float() / 255.0
            
        # WebDataset's .decode("rgb") returns (N, H, W, C) format — channels last.
        # PyTorch Conv2d expects (N, C, H, W) format — channels first.
        # .permute() rearranges dimensions WITHOUT copying data (just changes strides).
        # .contiguous() copies to a new contiguous memory block, required because
        # permute creates a non-contiguous view that crashes during backpropagation.
        if images.shape[-1] == 3:  # Channels-last detected
            images = images.permute(0, 3, 1, 2).contiguous()
            
        # Forward pass: images → model → logits
        outputs = self(images)
        
        # Cross-entropy loss (combines log_softmax + nll_loss for numerical stability)
        loss = F.cross_entropy(outputs, labels)
        
        # Compute accuracy for monitoring (not used for gradient computation)
        preds = torch.argmax(outputs, dim=1)      # Index of highest logit = predicted class
        acc = (preds == labels).float().mean()     # Fraction of correct predictions
        
        # self.log() records metrics to the progress bar and logger (CSVLogger/TensorBoard)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss  # Lightning calls loss.backward() on this

    def test_step(self, batch, batch_idx):
        """
        Evaluation step — identical to training_step but WITHOUT gradient computation.
        Lightning automatically wraps this in torch.no_grad() for efficiency.
        """
        images, labels = batch
        if images.dtype == torch.uint8:
            images = images.float() / 255.0
        if images.shape[-1] == 3:
            images = images.permute(0, 3, 1, 2).contiguous()
            
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean()
        self.log('test_loss', loss, prog_bar=True)
        self.log('test_acc', acc, prog_bar=True)

    def configure_optimizers(self):
        """
        Returns the optimizer. AdamW = Adam + decoupled Weight Decay.
        
        Learning rate 1e-3 is a safe default for Adam-family optimizers.
        In production, you'd use a learning rate scheduler (cosine annealing,
        one-cycle, warm-up + decay) for better convergence.
        """
        return optim.AdamW(self.model.parameters(), lr=1e-3)

# ============================================================================
# LAUNCH TRAINING
# ============================================================================
# Trainer configuration:
#   max_epochs=1:           Single pass over the data (for lecture time constraints)
#   limit_train_batches=100: Only process 100 batches (25,600 images) per epoch
#   limit_test_batches=20:  Only evaluate on 20 batches (5,120 images)
#   accelerator='auto':     Automatically selects GPU (CUDA/MPS) if available
trainer = pl.Trainer(
    max_epochs=1,
    limit_train_batches=100,
    limit_test_batches=20,
    accelerator='auto'
)
lightning_model = TrafficLight(model)

print("🚀 Starting Training Loop over Streaming Datasets...")
trainer.fit(lightning_model, loader)

print("🧪 Starting Evaluation Phase...")
trainer.test(lightning_model, loader)